Irrelevant

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path

from ipynb.fs.full.DataGenerator import prepare_data, HyperspectralDataset

In [ ]:
seeds_path = Path('..') / 'home' / 'ARO.local' / 'collaboration' / 'sagi-tomer-collab' / 'Normalized_Tomato_Seeds'
healthy_dir = seeds_path / 'Healthy'
infected_dir = seeds_path / 'Infected'

n = 10
bands = [100, 300, 500]

height = np.load(seeds_path / 'normalization_parameters' / 'max_height.npy').item() + 2
width = np.load(seeds_path / 'normalization_parameters' / 'max_width.npy').item() + 2
shape = (height, width, len(bands))

(train_files, train_labels), (val_files, val_labels), (test_files, test_labels) = prepare_data(str(healthy_dir), str(infected_dir), n, n)

train_data = HyperspectralDataset(train_files, train_labels, bands, shape).get_dataset(batch_size=1, shuffle=True)
val_data = HyperspectralDataset(val_files, val_labels, bands, shape).get_dataset(batch_size=1)
test_data = HyperspectralDataset(test_files, test_labels, bands, shape).get_dataset(batch_size=1)

In [4]:
def build_hyperspectral_cnn(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

In [5]:
model = build_hyperspectral_cnn(shape)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    steps_per_epoch=len(train_files),
    validation_steps=len(val_files),
    callbacks=callbacks
)

Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.4397 - loss: 0.7074 - val_accuracy: 0.5000 - val_loss: 0.6709 - learning_rate: 0.0010
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7589 - loss: 0.6374 - val_accuracy: 0.5000 - val_loss: 0.5730 - learning_rate: 0.0010
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8674 - loss: 0.4697 - val_accuracy: 1.0000 - val_loss: 0.3523 - learning_rate: 0.0010
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9277 - loss: 0.2121 - val_accuracy: 1.0000 - val_loss: 0.1836 - learning_rate: 0.0010
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8646 - loss: 0.2051 - val_accuracy: 1.0000 - val_loss: 0.2669 - learning_rate: 0.0010
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0646 - val_accuracy: 1.0000 - val_loss: 0.0467 - learning_rate: 0.0010
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.1107 - val_accuracy:

In [6]:
test_loss, test_acc = model.evaluate(test_data, steps=len(test_files))
print(f"Test Accuracy: {test_acc:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 6.5797e-05 
Test Accuracy: 1.0000
